# Jupyter + BSL fixture acceptance

Технический live-smoke MAIN/CAPTURE, включая mixed CAPTURE cells.

In [ ]:
CHUNK_SIZE = 128

In [ ]:
from datetime import datetime
from decimal import Decimal
import json
import os
from pathlib import Path

from IPython.display import JSON
from onec_runtime.capture_source import CapturePointRequest
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import RuntimeSessionConfig
from onec_runtime.value_materialization import ONEC_UNDEFINED
from onec_runtime_jupyter import InteractiveRuntimeSession
from onec_runtime_jupyter.extension import NotebookDisplayConfig

CALLEE_CAPTURE_FRAGMENT = 'ЛокальныйСчетчик = ЛокальныйСчетчик + 2;'
CALLER_CAPTURE_FRAGMENT = 'ЗначениеИзСтека = РезультатПодчиненного.Счетчик;'

workspace = Path(os.environ["ONEC_RUNTIME_WORKSPACE"])
runtime_config = RuntimeConfig(
    workspace=workspace,
    platform_bin=Path(os.environ["ONEC_RUNTIME_PLATFORM_BIN"]),
    connection_string=os.environ["ONEC_RUNTIME_CONNECTION_STRING"],
    username="",
)
fixture = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime_config,
        Path(os.environ["ONEC_RUNTIME_EVIDENCE_DIR"]),
        CHUNK_SIZE,
    ),
    shell=get_ipython(),
    display=NotebookDisplayConfig.diagnostic(),
)
fixture.configure_capture_source(
    "jupyter_fixture",
    Path(os.environ["ONEC_RUNTIME_FIXTURE_SOURCE"]),
)
Path(os.environ["ONEC_RUNTIME_PROCESS_SNAPSHOT"]).write_text(
    json.dumps(fixture.owned_process_snapshot(), ensure_ascii=False),
    encoding="utf-8",
)
python_sentinel = "alive"

In [ ]:
%bsl_status

In [ ]:
%%bsl
Счетчик = 5;
Вложенные = Новый Структура;
Числа = Новый Массив;
Числа.Добавить(3);
Числа.Добавить(5);
Вложенные.Вставить("Числа", Числа);
Вложенные.Вставить("Пусто", Неопределено);
Вложенные.Вставить("Флаг", Истина);
Вложенные.Вставить("Момент", Дата(2026, 8, 27, 12, 30, 0));
ТаблицаДанных = Новый ТаблицаЗначений;
ТаблицаДанных.Колонки.Добавить("Код");
ТаблицаДанных.Колонки.Добавить("Сумма");
СтрокаДанных = ТаблицаДанных.Добавить();
СтрокаДанных.Код = "A";
СтрокаДанных.Сумма = 10;
СтрокаДанных = ТаблицаДанных.Добавить();
СтрокаДанных.Код = "B";
СтрокаДанных.Сумма = 20;
СтрокаДанных = ТаблицаДанных.Добавить();
СтрокаДанных.Код = "C";
СтрокаДанных.Сумма = 30;
СостояниеМетода = Новый Структура("Результат", 0);
СостояниеMixed = Новый Структура("Результат", 0);
Сообщить("main-ready");
Результат = Счетчик;

In [ ]:
main_scalar = Счетчик.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
main_nested_first = Вложенные.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
main_nested_second = Вложенные.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
main_rows = ТаблицаДанных.head(2).to_df(chunk_size=128)
assert main_scalar == Decimal("5")
assert type(main_nested_first["Флаг"]) is bool and main_nested_first["Флаг"] is True
assert main_nested_first["Момент"] == datetime(2026, 8, 27, 12, 30, 0)
assert main_nested_first["Пусто"] is ONEC_UNDEFINED
assert main_nested_first["Числа"] == [Decimal("3"), Decimal("5")]
assert main_nested_second == main_nested_first
assert list(main_rows.columns) == ["Код", "Сумма"]
assert main_rows["Код"].tolist() == ["A", "B"]
assert [int(value) for value in main_rows["Сумма"]] == [10, 20]
main_table_proxy = ТаблицаДанных
JSON({"status": "PASS", "nested_numbers": [3, 5], "rows": 2})

In [ ]:
%%bsl
Процедура FixtureMainWorker(Состояние, Значение)
    Состояние.Результат = Значение * 2;
КонецПроцедуры

In [ ]:
main_rows_repeat = main_table_proxy.head(2).to_df(chunk_size=128)
assert main_rows_repeat["Код"].tolist() == ["A", "B"]
JSON({"status": "PASS", "same_generation": True, "rows": 2})

In [ ]:
%%bsl
FixtureMainWorker(СостояниеМетода, 6);
MainMethodResult = СостояниеМетода.Результат;
Результат = MainMethodResult;

In [ ]:
%%bsl
Процедура FixtureMixedMain(Состояние, Значение)
    Состояние.Результат = Значение * 3;
КонецПроцедуры;
FixtureMixedMain(СостояниеMixed, 7);
MainMixedResult = СостояниеMixed.Результат;
Результат = MainMixedResult;

In [ ]:
%%bsl
Счетчик = 999;
ВызватьИсключение "fixture-main-error";

In [ ]:
%%bsl
Сообщить("main-recovery:" + Счетчик + ":" + MainMethodResult + ":" + MainMixedResult);
Результат = "" + Счетчик + "|" + MainMethodResult + "|" + MainMixedResult;

In [ ]:
capture_points = fixture.resolve_capture_points((
    CapturePointRequest("callee", source_fragment=CALLEE_CAPTURE_FRAGMENT),
    CapturePointRequest("caller", source_fragment=CALLER_CAPTURE_FRAGMENT),
))
capture_bindings = fixture.verify_capture_points(capture_points)
fixture.configure_capture_points(
    tuple(binding.location for binding in capture_bindings)
)
capture_points

In [ ]:
%%bsl
РезультатFixture = JupyterBslFixtureCallerServer.ВыполнитьСценарий();
Результат = РезультатFixture;

In [ ]:
%%bsl
CaptureCounter = КонтекстОтладки.ЛокальныйСчетчик;
CaptureNested = КонтекстОтладки.ЛокальнаяСтруктура;
CaptureTable = КонтекстОтладки.ЛокальнаяТаблица;
CaptureWorkerState = Новый Структура("Результат", 0);
РезультатИнструкции = CaptureCounter;

In [ ]:
capture_scalar = CaptureCounter.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
capture_nested_first = CaptureNested.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
capture_nested_second = CaptureNested.materialize(
    max_depth=8, max_items=32, max_bytes=65536
)
capture_rows = CaptureTable.head(2).to_df(chunk_size=128)
assert capture_scalar == Decimal("10")
assert capture_nested_first["Флаг"] is True
assert capture_nested_first["Момент"] == datetime(2026, 8, 27, 12, 30, 0)
assert capture_nested_first["Пусто"] is ONEC_UNDEFINED
assert capture_nested_first["Числа"] == [Decimal("3"), Decimal("5")]
assert capture_nested_second == capture_nested_first
assert list(capture_rows.columns) == ["Код", "Сумма"]
assert capture_rows["Код"].tolist() == ["A", "B"]
assert [int(value) for value in capture_rows["Сумма"]] == [10, 20]
capture_table_proxy = CaptureTable
JSON({"status": "PASS", "counter": 10, "rows": 2})

In [ ]:
%%bsl
Процедура FixtureCaptureWorker(Состояние, Значение)
    Состояние.Результат = Значение + 5;
КонецПроцедуры

In [ ]:
capture_rows_repeat = capture_table_proxy.head(2).to_df(chunk_size=128)
assert capture_rows_repeat["Код"].tolist() == ["A", "B"]
JSON({"status": "PASS", "same_generation": True, "rows": 2})

In [ ]:
%%bsl
FixtureCaptureWorker(CaptureWorkerState, КонтекстОтладки.ЛокальныйСчетчик);
CaptureMethodResult = CaptureWorkerState.Результат;
РезультатИнструкции = CaptureMethodResult;

In [ ]:
%%bsl
Процедура FixtureMixedCapture(Состояние, Значение)
    Состояние.Результат = Значение * 2;
КонецПроцедуры;
CaptureMixedState = Новый Структура("Результат", 0);
FixtureMixedCapture(CaptureMixedState, КонтекстОтладки.ЛокальныйСчетчик);
КонтекстОтладки.ЛокальныйMixed = CaptureMixedState.Результат;
CaptureMixedResult = КонтекстОтладки.ЛокальныйMixed;
РезультатИнструкции = CaptureMixedResult;

In [ ]:
%%bsl
Процедура FixtureMixedCaptureAfterError(Состояние, Значение)
    Состояние.Результат = Значение + 1;
КонецПроцедуры;
КонтекстОтладки.ЛокальныйСчетчик = 901;
ОшибкаMixedCapture = 1 / 0;

In [ ]:
%%bsl
CaptureMixedErrorState = Новый Структура("Результат", 0);
FixtureMixedCaptureAfterError(CaptureMixedErrorState, КонтекстОтладки.ЛокальныйСчетчик);
CaptureMixedErrorRecovery = CaptureMixedErrorState.Результат;
РезультатИнструкции = CaptureMixedErrorRecovery;

In [ ]:
%%bsl
КонтекстОтладки.ЛокальныйСчетчик = 900;
ВызватьИсключение "fixture-capture-error";

In [ ]:
%%bsl
Сообщить("capture-recovery:" + КонтекстОтладки.ЛокальныйСчетчик);
РезультатИнструкции = КонтекстОтладки.ЛокальныйСчетчик;

In [ ]:
%%bsl
КонтекстОтладки.ЛокальныйСчетчик = 40;
КонтекстОтладки.ЛокальнаяСтруктура.Метка = "после";
РезультатИнструкции = КонтекстОтладки.ЛокальныйСчетчик;

In [ ]:
get_ipython().run_line_magic(
    "bsl_resume", "ЛокальныйСчетчик ЛокальнаяСтруктура"
)

In [ ]:
%%bsl
StackValue = КонтекстОтладки.РезультатПодчиненного.Счетчик;
StackMarker = КонтекстОтладки.РезультатПодчиненного.Метка;
StackMixed = КонтекстОтладки.РезультатПодчиненного.Mixed;
Сообщить("capture-stack:" + StackMarker + ":mixed=" + StackMixed);
РезультатИнструкции = StackValue;

In [ ]:
%bsl_resume

In [ ]:
final_status = fixture.status()
assert final_status.state.value == "completed"
assert python_sentinel == "alive"
assert int(main_scalar) == 5
assert int(capture_scalar) == 10
JSON({
    "status": "PASS",
    "main_proxy_rows": len(main_rows),
    "capture_proxy_rows": len(capture_rows),
    "python_sentinel": python_sentinel,
})

In [ ]:
fixture.close()
fixture.close()